In [1]:
import glob
import os

root = "data/raw/humaid/events_set1"

for name in ["Readme.txt", "Licensing.txt"]:
    p = os.path.join(root, name)
    print("=====", name)
    print(open(p, encoding="utf-8", errors="ignore").read() if os.path.exists(p) else "not found")

files = [f for f in glob.glob(f"{root}/**/*", recursive=True)
         if os.path.isfile(f) and not os.path.basename(f).startswith(".")]
print(len(files), "files")
for f in files[:30]:
    print(f, os.path.getsize(f))

===== Readme.txt
Description of the dataset
The HumAID Twitter dataset consists of several thousands of manually annotated tweets that have been collected during nineteen major natural disaster events including earthquakes, hurricanes, wildfires, and floods, which happened during 2016 to 2019 across different parts of the World. It is the largest social media dataset (~77K) for crisis informatics so far. The annotations consist of following humanitarian categories.

Humanitarian categories
  * Caution and advice
  * Displaced people and evacuations
  * Dont know cant judge
  * Infrastructure and utility damage
  * Injured or dead people
  * Missing or found people
  * Not humanitarian
  * Other relevant information
  * Requests or urgent needs
  * Rescue volunteering or donation effort
  * Sympathy and support

Data format and directories
The data directory contains a sample dataset:

*  events/ This directory contains sub-directories for each event. In which each event directory conta

In [2]:
import glob
import os
import pandas as pd

root = "data/raw/humaid/events_set1"
keep = ["harvey", "irma", "maria", "matthew", "floods", "idai"]

frames = []
for f in glob.glob(f"{root}/**/*.tsv", recursive=True):
    name = os.path.basename(f).lower()
    if not name.endswith(("_train.tsv", "_dev.tsv", "_test.tsv")):
        continue
    if not any(k in name for k in keep):
        continue
    d = pd.read_csv(f, sep="\t", quoting=3, on_bad_lines="skip")
    d["event"] = name.rsplit("_", 1)[0]
    d["split"] = name.rsplit("_", 1)[1].replace(".tsv", "")
    frames.append(d)

tw = pd.concat(frames, ignore_index=True)
print(tw.shape)
print(tw["event"].value_counts())
print(tw["class_label"].value_counts())
tw.sample(8, random_state=1)[["tweet_text", "class_label"]]

(32678, 5)
event
hurricane_irma_2017       9399
hurricane_harvey_2017     9112
hurricane_maria_2017      7278
cyclone_idai_2019         3933
hurricane_matthew_2016    1654
maryland_floods_2018       742
srilanka_floods_2017       560
Name: count, dtype: int64
class_label
rescue_volunteering_or_donation_effort    8977
other_relevant_information                6621
infrastructure_and_utility_damage         5153
sympathy_and_support                      2901
injured_or_dead_people                    2653
displaced_people_and_evacuations          1669
caution_and_advice                        1597
not_humanitarian                          1548
requests_or_urgent_needs                  1400
missing_or_found_people                    159
Name: count, dtype: int64


,tweet_text,class_label
7189,Texas governor urges evacuation ahead of Hurri...,displaced_people_and_evacuations
16119,Hurricane Irma is on track to hit Miami early ...,injured_or_dead_people
2880,"A city of 500,000 inhabitants destroyed. We ha...",rescue_volunteering_or_donation_effort
7192,Budweiser brewery fills beer cans with water t...,sympathy_and_support
3243,Happening now: just offloading more mama and f...,rescue_volunteering_or_donation_effort
195,"#CycloneIdai Today, President @edmnangagwa is ...",sympathy_and_support
11261,ATF agents continue working alongside @houston...,rescue_volunteering_or_donation_effort
183,Dear World Leaders: Please make sure all donat...,rescue_volunteering_or_donation_effort


In [3]:
import re

HAZARD = ["infrastructure_and_utility_damage"]
NON_HAZARD = ["not_humanitarian", "sympathy_and_support",
              "rescue_volunteering_or_donation_effort", "other_relevant_information"]
# dropped as ambiguous: injured_or_dead_people, displaced_people_and_evacuations,
# caution_and_advice, requests_or_urgent_needs, missing_or_found_people

def to_safety(label):
    if label in HAZARD:
        return "hazard"
    if label in NON_HAZARD:
        return "non_hazard"
    return None

def clean_tweet(t):
    t = str(t)
    t = re.sub(r"https?://\S+", " ", t)      # links
    t = re.sub(r"@\w+", " ", t)              # @mentions
    t = re.sub(r"^rt\s+", "", t, flags=re.I) # retweet marker
    t = t.replace("#", " ")
    return re.sub(r"\s+", " ", t).strip()

tw["safety_label"] = tw["class_label"].map(to_safety)
sf = tw.dropna(subset=["safety_label"]).copy()
sf["text"] = sf["tweet_text"].apply(clean_tweet)
sf = sf[sf["text"].str.len() > 15]
sf = sf.drop_duplicates(subset="text")       # retweets and copies
print(sf.shape)
sf["safety_label"].value_counts()

(25196, 7)


safety_label
non_hazard    20043
hazard         5153
Name: count, dtype: int64

In [4]:
pd.set_option("display.max_colwidth", 200)
haz = sf[sf["safety_label"] == "hazard"]
display(haz.sample(8, random_state=2)[["text", "event"]])

kw = {
    "flood":         "flood|water|submerged|inundat",
    "power_outage":  "power|electric|outage|blackout|grid",
    "road_blockage": "road|bridge|highway|tree|street|blocked|debris",
}
for k, pat in kw.items():
    print(k, haz["text"].str.contains(pat, case=False).sum(), "of", len(haz))

,text,event
4126,"WATCH: Hurricane Harvey could cause an estimated $1-2 billion in property damage just from wind and storm surge, says.",hurricane_harvey_2017
6203,"Residents of Texas HOA stranded after road, bridge destroyed in flood via HurricaneHarvey",hurricane_harvey_2017
22891,"More than a week after Hurricane Maria hit Puerto Rico, Marines are still clearing roads",hurricane_maria_2017
8713,"Floodgates to open at Dam B harvey overflow &amp; the damage is seemingly insurmountable-but as Texans, we are strong",hurricane_harvey_2017
5720,"Hurricane Harvey will reach Category 3 strength this aftn. Flooding, major damage &amp; power outages. Harvey",hurricane_harvey_2017
19193,: Trumps Caribbean mansion probably destroyed by Hurricane Irma,hurricane_irma_2017
31655,"Footage of flooding in Ellicott City, Maryland courtesy of Zach Landow.",maryland_floods_2018
28983,: Heres a few photos for you showing some of the damage Hurricane Maria has done on Puerto Rico.,hurricane_maria_2017


flood 1002 of 5153
power_outage 467 of 5153
road_blockage 351 of 5153


In [5]:
os.makedirs("data/processed", exist_ok=True)
sf.to_csv("data/processed/humaid_safety.csv", index=False)
print("Saved", len(sf), "rows")

Saved 25196 rows
